In [ ]:
import numpy as np

path = "2025_combined_mlar_header.txt"

data = np.genfromtxt(path,
                    delimiter = "|",
                    names = True,
                    # dtype = None,
                    missing_values = ("NA", "")
                    ,max_rows = 9998
                    )

In [ ]:
import numpy as np

path = "2025_combined_mlar_header.txt"

selected_columns = [
    "state_code",
    "loan_type",
    "action_taken",
    "debt_to_income_ratio"
]

with open(path, "r", encoding="utf-8-sig") as f:
    header = f.readline().strip().split("|")

column_indices = [
    header.index(col) for col in selected_columns
]

extracted_data = np.genfromtxt(
    path,
    delimiter="|",
    skip_header=1,
    usecols=column_indices,
    max_rows=9998,
    dtype=str,
    encoding="utf-8"
)

print(extracted_data[:10])
print(extracted_data.shape)
print(type(extracted_data))

In [ ]:
import numpy as np

column_names = [
    "state",
    "loan_type",
    "action_taken",
    "debt_to_income"
]

missing_mask = np.isin(
    np.char.strip(extracted_data),
    ["", "NA", "nan", "NaN"]
)

for i, name in enumerate(column_names):
    print(name, "missing:", np.sum(missing_mask[:, i]))

In [ ]:
import numpy as np
import re

cleaned_data = extracted_data.astype(object).copy()

dti_original = cleaned_data[:, 3].copy()

def convert_dti(value):
    value = str(value).strip()

    if value.upper() in ("", "NA", "NAN"):
        return np.nan

    try:
        return float(value)
    except ValueError:
        pass

    match = re.fullmatch(
        r"(\d+(?:\.\d+)?)%-<(\d+(?:\.\d+)?)%",
        value
    )

    if match:
        lower = float(match.group(1))
        upper = float(match.group(2))
        return (lower + upper) / 2

    if value == ">60%":
        return 65.0

    return np.nan

dti_numeric = np.array([
    convert_dti(x) for x in dti_original
], dtype=float)

missing = np.array([
    str(x).strip().upper() in ("", "NA", "NAN")
    for x in dti_original
])

valid = ~np.isnan(dti_numeric)
indices = np.arange(len(dti_numeric))

print("Missing before:", missing.sum())

if valid.sum() >= 2:
    inside = (
        missing &
        (indices > indices[valid][0]) &
        (indices < indices[valid][-1])
    )

    interpolated = np.interp(
        indices[inside],
        indices[valid],
        dti_numeric[valid]
    )

    cleaned_data[inside, 3] = [
        str(round(x, 2)) for x in interpolated
    ]

remaining = np.array([
    str(x).strip().upper() in ("", "NA", "NAN")
    for x in cleaned_data[:, 3]
])

print("Missing after:", remaining.sum())

print("\nOriginal DTI:")
print(dti_original[:10])

print("\nCleaned DTI:")
print(cleaned_data[:10, 3])

print("\nData shape:", cleaned_data.shape)
print("Data type:", type(cleaned_data))

In [ ]:
other_columns = extracted_data[:, :3]

final_data = np.column_stack((
    other_columns,
    dti_interpolated
))

print(final_data[:10])
print(final_data.shape)
print(type(final_data))

In [ ]:
for i, name in enumerate([
    "state", "loan_type", "action_taken", "debt_to_income"
]):
    missing = np.isin(
        np.char.strip(final_data[:, i]),
        ["", "NA", "nan", "NaN"]
    )
    print(name, ":", np.sum(missing))

In [ ]:
import numpy as np

reordered_data = cleaned_data[:, [3, 0, 1, 2]]

print(reordered_data[:10])
print(reordered_data.shape)
print(type(reordered_data))

In [ ]:
cleaned_data[:, [3, 0, 1, 2]]

In [ ]:
import numpy as np

column_names = ["DTI", "State", "LoanType", "action_taken"]

for i, name in enumerate(column_names):
    values = np.char.upper(
        np.char.strip(reordered_data[:, i].astype(str))
    )

    missing = np.isin(values, ["", "NA", "NAN", "NONE", "NULL"])

    print(name, "missing values:", np.sum(missing))

In [ ]:
print(data.shape)
print(data.dtype.names)

In [ ]:
selected_data = data[
    ["state_code",
     "loan_type",
     "action_taken",
     "debt_to_income_ratio"]
    ]